{% include nav/toolkits/bathroom/menu.html %}

<head>
    <link href="https://fonts.googleapis.com/icon?family=Material+Icons" rel="stylesheet">
</head>

<style>
    .container {
        position: relative;
        margin-top: 10px !important;
    }
    body {
        color: black; 
    }
    .header-container {
        text-align: center;
        margin-top: 20px;
    }
    .header-container h1 {
        font-size: 28px;
        color: white;
    }
</style>

<script type="module">
    import { javaURI, fetchOptions } from '{{ site.baseurl }}/assets/js/api/config.js';

    document.addEventListener("DOMContentLoaded", function () {
        document.getElementById("requestPassBtn").addEventListener("click", requestHallPass);
        document.getElementById("checkinBtn").addEventListener("click", checkinHallPass);
        loadSavedData(); // Load saved teacher, period, and activity on page load
    });

    function loadSavedData() {
        const storedTeacherFirstName = localStorage.getItem("teacherFirstName");
        const storedTeacherLastName = localStorage.getItem("teacherLastName");
        const storedPeriod = localStorage.getItem("period");
        const storedActivity = localStorage.getItem("activity");

        if (storedTeacherFirstName && storedTeacherLastName && storedPeriod && storedActivity) {
            document.getElementById("active-teacher").textContent = `${storedTeacherFirstName} ${storedTeacherLastName}`;
            document.getElementById("active-period").textContent = storedPeriod;
            document.getElementById("active-activity").textContent = storedActivity;
            
            // Hide the input form and show the hall pass request section
            document.getElementById("teacher-form").style.display = "none"; 
            document.getElementById("pass-form").style.display = "block"; 
        } else {
            document.getElementById("teacher-form").style.display = "block"; // Show input form
            document.getElementById("pass-form").style.display = "none"; // Hide hall pass request
        }
    }

    async function requestHallPass() {
        const teacherFirstName = document.getElementById("teacherFirstName").value.trim();
        const teacherLastName = document.getElementById("teacherLastName").value.trim();
        const period = document.getElementById("period").value;
        const activity = document.getElementById("activity").value;
        const resultDiv = document.getElementById("confirm-message");

        if (!teacherFirstName || !teacherLastName || !period || !activity) {
            alert("Please fill out all fields.");
            return;
        }

        // Save all inputs to local storage
        localStorage.setItem("teacherFirstName", teacherFirstName);
        localStorage.setItem("teacherLastName", teacherLastName);
        localStorage.setItem("period", period);
        localStorage.setItem("activity", activity);

        resultDiv.innerHTML = ""; // Clear previous results

        const requestBody = {
            userName: "toby",
            teacherFirstName: teacherFirstName,
            teacherLastName: teacherLastName,
            period: period,
            activity: activity
        };

        try {
            const response = await fetch(`${javaURI}/api/hallpass/request`, {
                method: "POST",
                headers: { "Content-Type": "application/json" },
                body: JSON.stringify(requestBody)
            });

            if (!response.ok) {
                throw new Error(`HTTP error! Status: ${response.status}`);
            }

            document.getElementById("teacher-form").style.display = "none"; // Hide form after save
            document.getElementById("pass-form").style.display = "none";
            document.getElementById("active-pass-container").style.display = "block";
            resultDiv.innerHTML = `<span style="color: green;font-size: 20px;">Success! Your hall pass has been processed.</span>`;
        } catch (error) {
            console.error("Error requesting hall pass:", error);
        }
    }

    async function checkinHallPass() {
        try {
            const response = await fetch(`${javaURI}/api/hallpass/checkout?email=toby`, {
                method: "POST",
                headers: { "Content-Type": "application/json" }
            });

            const resultDiv = document.getElementById("confirm-message");
            resultDiv.innerHTML = ""; // Clear previous results

            if (!response.ok) {
                throw new Error(`HTTP error! Status: ${response.status}`);
            }

            document.getElementById("pass-form").style.display = "none";
            document.getElementById("active-pass-container").style.display = "none";
            document.getElementById("confirmation-container").style.display = "block";
            
            resultDiv.innerHTML = `<span style="color: green;font-size: 20px;">Success! Your hall pass visit has been recorded.</span>`;
        } catch (error) {
            console.error("Error checking in hall pass:", error);
        }
    }
</script>

<body>
    <div class="header-container">
        <h1 style="color:#0947ab;">Welcome to Hall Pass Reques</h1>
        <img src="https://github.com/user-attachments/assets/0f91ecf3-72a3-4712-b0ba-8d0f8ca55bb3">
    </div>
    <div class="container bg-primary text-black">
        <div id="confirmation-container" class="bg-light rounded" style="display: none;">
            <p id="confirm-message"></p>
        </div>
        <div id="error-container" class="alert alert-danger" role="alert" style="display: none;">
            <p id="error-message"></p>
        </div> 
        <!-- Active Hall Pass Section -->
        <div id="active-pass-container" class="bg-light rounded" style="display: none;">
            <h2>You have an active hall pass. Please check-in to end session</h2>
            <p><strong>Teacher:</strong> <span id="active-teacher"></span></p>
            <p><strong>Activity:</strong> <span id="active-activity"></span></p>
            <p><strong>Period:</strong> <span id="active-period"></span></p>
            <button id="checkinBtn" class="btn btn-primary mt-3">Check-in Hall Pass</button>            
        </div>
        <!-- Teacher, Period, and Activity Input Form -->
        <div id="teacher-form" class="bg-light rounded" style="display: none;">
            <h2>Enter Information</h2>
            <div class="form-group">
                <label for="teacherFirstName">Teacher First Name:</label>
                <input type="text" id="teacherFirstName" name="teacherFirstName" class="form-control" required>
            </div>
            <div class="form-group">
                <label for="teacherLastName">Teacher Last Name:</label>
                <input type="text" id="teacherLastName" name="teacherLastName" class="form-control" required>
            </div>
            <div class="form-group">
                <label for="period">Period:</label>
                <select id="period" class="form-control">
                    <option value="1">1</option>
                    <option value="2">2</option>
                    <option value="3">3</option>
                    <option value="4">4</option>
                    <option value="5">5</option>
                </select>
            </div>
            <div class="form-group">
                <label for="activity">Activity:</label>
                <select id="activity" class="form-control">
                    <option value="bathroom">Bathroom</option>
                    <option value="library">Library</option>
                    <option value="other">Other</option>
                </select>
            </div>
        </div>
        <!-- Request Pass Button -->
        <button id="requestPassBtn" class="btn btn-primary mt-3">Request Hall Pass</button>         
    </div>
</body>

 
  


 